In [21]:
!pip install --upgrade --quiet gcloud

In [22]:
!gcloud auth application-default login


/bin/bash: line 1: gcloud: command not found


In [17]:
# Use the environment variable if the user doesn't provide Project ID.
import os

PROJECT_ID = "flash-landing-473808-v4" 
LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

# Initialize Vertex AI
import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION)

In [18]:
# for data processing
import numpy as np
import pandas as pd
import seaborn as sns

pd.options.mode.chained_assignment = None  # default='warn'

# for similarity calculation
from sklearn.metrics.pairwise import cosine_similarity


# vertex ai sdk
from vertexai.vision_models import Image as VMImage
from vertexai.vision_models import MultiModalEmbeddingModel

In [19]:
mm_embedding_model = MultiModalEmbeddingModel.from_pretrained("multimodalembedding")

/home/quang/miniconda3/envs/paraline/lib/python3.10/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


GoogleAuthError: 
Unable to authenticate your request.
Depending on your runtime environment, you can complete authentication by:
- if in local JupyterLab instance: `!gcloud auth login` 
- if in Colab:
    -`from google.colab import auth`
    -`auth.authenticate_user()`
- if in service account or other: please follow guidance in https://cloud.google.com/docs/authentication

In [ ]:
def get_text_embedding(
    text: str = "banana muffins",
    dimension: int | None = 1408,
) -> list[float]:
    embedding = mm_embedding_model.get_embeddings(
        contextual_text=text,
        dimension=dimension,
    )
    return embedding.text_embedding


def get_image_embedding(
    image_path: str,
    dimension: int | None = 1408,
) -> list[float]:
    image = VMImage.load_from_file(image_path)
    embedding = mm_embedding_model.get_embeddings(
        image=image,
        dimension=dimension,
    )
    return embedding.image_embedding



In [ ]:
text_emb = get_text_embedding(text="What is life?")
print("length of embedding: ", len(text_emb))
print("First five values are: ", text_emb[:5])

In [ ]:
text = [
    "i really enjoyed the movie last night",
    "so many amazing cinematic scenes yesterday",
    "had a great time writing my Python scripts a few days ago",
    "huge sense of relief when my .py script finally ran without error",
    "O Romeo, Romeo, wherefore art thou Romeo?",
]

df = pd.DataFrame(text, columns=["text"])
df

In [ ]:
df["embedding"] = df.apply(lambda x: get_text_embedding(x.text), axis=1)
df

In [ ]:
cos_sim_array = cosine_similarity(list(df.embedding.values))

# display as DataFrame
df = pd.DataFrame(cos_sim_array, index=text, columns=text)
df

In [ ]:
ax = sns.heatmap(df, annot=True, cmap="crest")
ax.xaxis.tick_top()
ax.set_xticklabels(text, rotation=90)

In [ ]:
# Image embeddings with default 1408 dimension
image_path = "gs://github-repo/embeddings/getting_started_embeddings/gms_images/GGOEACBA104999.jpg"
print(get_public_url_from_gcs(image_path))

image_emb = get_image_embedding(
    image_path=image_path,
)
print("length of embedding: ", len(image_emb))
print("First five values are: ", image_emb[:5])

In [ ]:
# get product list with pre-computed image embeddings
product_image_list = pd.read_csv(
    "https://storage.googleapis.com/github-repo/embeddings/getting_started_embeddings/image_data_with_embeddings.csv"
)
product_image_list.head()

In [ ]:
# calc_scores for a text query
query_emb = get_text_embedding("something related to dinosaurs theme")
print_similar_images(query_emb, product_image_list)

In [ ]:
query_emb = get_text_embedding("Socks in checkered patterns")
print_similar_images(query_emb, product_image_list)